# Download LLM Model to Snowflake Stage

Downloads the **Qwen2.5-1.5B-Instruct GGUF** model (~1 GB) from HuggingFace and uploads it to `@TRAVEL_DEMO.AGENTS.LLM_MODELS` for use by the llama.cpp server in the `TRAVEL_ORCHESTRATOR` SPCS service.

## Prerequisites

- `SNOWFLAKE_SETUP.sql` must have been run (the `@LLM_MODELS` stage must exist)
- This notebook must run on the **TRAVEL_DEMO_POOL** compute pool (Container Runtime):
  - In Snowflake Workspace: open notebook → **Session** panel → **Compute** → select `TRAVEL_DEMO_POOL`
  - The SPCS node provides sufficient local disk (~tens of GBs) and direct internet egress — no External Access Integration is required

Run all cells in order. The upload typically takes 1–3 minutes depending on network speed.

In [ ]:
from snowflake.snowpark.context import get_active_session
import requests
import os

session = get_active_session()
print(f"Database : {session.get_current_database()}")
print(f"Schema   : {session.get_current_schema()}")
print(f"Role     : {session.get_current_role()}")
print(f"Warehouse: {session.get_current_warehouse()}")

In [ ]:
MODEL_URL  = 'https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct-GGUF/resolve/main/qwen2.5-1.5b-instruct-q4_k_m.gguf'
STAGE_PATH = '@TRAVEL_DEMO.AGENTS.LLM_MODELS'

filename   = MODEL_URL.split('/')[-1]
local_path = f'/tmp/{filename}'

print(f'Downloading {filename} ...')
response = requests.get(MODEL_URL, stream=True, timeout=600)
response.raise_for_status()

total_bytes = int(response.headers.get('content-length', 0))
downloaded  = 0

with open(local_path, 'wb') as f:
    for chunk in response.iter_content(chunk_size=4 * 1024 * 1024):  # 4 MB chunks
        f.write(chunk)
        downloaded += len(chunk)
        if total_bytes:
            pct = downloaded * 100 // total_bytes
            print(f'\r  {pct}%  ({downloaded // (1024*1024)} / {total_bytes // (1024*1024)} MB)', end='', flush=True)

print(f'\nDownload complete — {downloaded // (1024*1024)} MB saved to {local_path}')

In [ ]:
print(f'Uploading {filename} to {STAGE_PATH} ...')
result = session.file.put(
    f'file://{local_path}',
    STAGE_PATH,
    auto_compress=False,
    overwrite=True,
)
print(f'PUT result: {result}')

os.remove(local_path)
print('Local temp file removed.')

In [ ]:
# Verify the model file is present in the stage
session.sql(f'LIST {STAGE_PATH}').show()